# Bayesian Recursion, Monte-Carlo Integration & Importance Sampling

*Course 4 — Particle Filters, Part 1. Kalman filters assume Gaussian densities. When the true posterior is **non-Gaussian or multimodal**, we need a method that represents *any* density. This notebook builds the tools: the exact **Bayesian recursion**, **Monte-Carlo integration**, **importance sampling**, and representing a pdf as a **sum of impulses**.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 The Exact Bayesian Filtering Recursion

- The fully general solution to recursive estimation is two steps. **Prediction** (Chapman–Kolmogorov):

$$
f(x_k \mid \mathbb{Z}_{k-1}) = \int f(x_k \mid x_{k-1})\, f(x_{k-1}\mid\mathbb{Z}_{k-1})\, dx_{k-1}.
$$

  → Spread the previous posterior forward through the transition density — "where could the state move to, weighted by how likely each start was."

- **Measurement update** (Bayes' rule):

$$
f(x_k \mid \mathbb{Z}_k) = \frac{f(z_k \mid x_k)\, f(x_k\mid\mathbb{Z}_{k-1})}{f(z_k\mid\mathbb{Z}_{k-1})}.
$$

  → Reweight the prediction by the **likelihood** of the new measurement, then normalize. The denominator (the **evidence**) is just a constant that makes it integrate to 1.

- **→ Intuition:** the Kalman filter is the *special case* of this recursion when everything is linear-Gaussian (the integrals then have closed forms). In general these integrals have no closed form — that's the whole problem the particle filter solves.

### 🧩 Why It's Hard — The Curse of Dimensionality

- The recursion needs multidimensional integrals over the state space. **Brute-force numeric integration** (rectangular/trapezoidal rules) grids the space:

$$
\mu = \int g(x)f(x)\,dx \approx \sum_i g(x_i)f(x_i)\,\Delta x.
$$

  → Evaluate on a grid and sum. Fine in 1-D, but a grid of $M$ points per axis needs $M^n$ points in $n$ dimensions.

- **→ Intuition:** this is the **curse of dimensionality** — grid cost explodes exponentially with state dimension. Even coarse grids are hopeless beyond a few dimensions. We need a method whose cost doesn't blow up with $n$.

### 🧩 Monte-Carlo Integration

- Rewrite any expectation as an average over samples drawn from the density:

$$
\mu = \mathbb{E}[g(x)] = \int g(x)f(x)\,dx \approx \frac{1}{N}\sum_{i=1}^{N} g(x^{(i)}), \qquad x^{(i)} \sim f(x).
$$

  → Instead of gridding, **draw $N$ random samples** from $f$ and average $g$ over them. By the law of large numbers this converges to the true integral.

- The killer feature: the error shrinks like $\mathcal{O}(1/\sqrt{N})$ **independent of dimension $n$**.

  → Monte-Carlo **breaks the curse of dimensionality** — accuracy depends on the *number of samples*, not the *dimension of the space*. This is why particle filters can work where grid methods can't.

- **→ Intuition:** put your samples where the probability actually is (drawn from $f$), and you spend no effort on the vast empty regions a grid would waste points on.

### 🧩 Importance Sampling — When You Can't Sample from $f$

- Often we can't draw samples directly from $f(x)$. Draw from an easier **importance density** $q(x)$ instead and correct with weights:

$$
\mu = \int g(x)\,\frac{f(x)}{q(x)}\,q(x)\,dx \approx \frac{1}{N}\sum_{i=1}^{N} g(x^{(i)})\,w(x^{(i)}), \quad w(x^{(i)}) = \frac{f(x^{(i)})}{q(x^{(i)})}, \; x^{(i)}\sim q(x).
$$

  → Sample from a convenient $q$, then **re-weight** each sample by how much more (or less) likely it was under the true $f$ than under $q$. Samples from over-represented regions get down-weighted, and vice versa.

- **→ Intuition:** $q$ should overlap $f$'s important regions; the weights fix the mismatch. This is the mechanism that lets a particle filter *propose* particles cheaply and *correct* with the measurement likelihood.

### 🧩 Weight Normalization — Dodging the Evidence Integral

- The Bayes denominator $f(z_k\mid\mathbb{Z}_{k-1})$ is an ugly integral we'd rather not compute. If we use only the **unnormalized** posterior $\tilde{f} = c\,f$, Monte-Carlo estimates get biased by the unknown constant $c$ — but we can measure and cancel it.

- Using **unnormalized weights** $\tilde{w}^{(i)} = \tilde{f}(x^{(i)})/q(x^{(i)})$ and **normalizing** them:

$$
w^{*}(x^{(i)}) = \frac{\tilde{w}^{(i)}}{\sum_{j=1}^{N}\tilde{w}^{(j)}}, \qquad \mu \approx \sum_{i=1}^{N} g(x^{(i)})\,w^{*}(x^{(i)}).
$$

  → Divide each weight by the sum of all weights. The unknown constant $c$ appears in numerator and denominator and **cancels** — so we never compute the evidence integral.

- **→ Intuition:** normalization requires only a **1-D sum over particles**, no matter how high-dimensional the state is — cheap and exact. This single trick is what makes the particle filter practical.

### 🧩 Representing a pdf as a Sum of Impulses

- We can't store a general multidimensional pdf as a table (curse of dimensionality again). Represent it as a **weighted sum of Dirac impulses** at the sample locations:

$$
\hat{f}(x) = \sum_{i=1}^{N} w^{(i)}\,\delta\big(x - x^{(i)}\big).
$$

  → Store only the **particle locations $x^{(i)}$ and their weights $w^{(i)}$** — a tiny list — instead of a giant grid. Points cluster densely where probability is high, sparsely where it's low.

- The **sifting property** $\int g(x)\delta(x-x^{(i)})\,dx = g(x^{(i)})$ makes expectations trivial: $\int g(x)\hat f(x)\,dx = \sum_i w^{(i)}g(x^{(i)})$.

  → Integrating against impulses just samples $g$ at the particle locations — exactly the Monte-Carlo estimator. The impulse representation and MC integration are two views of the same idea.

- To **visualize** the (spiky) impulse pdf, replace each impulse with a smooth **kernel** "bump" $b_\sigma$ (a kernel density estimate):

$$
\hat{f}(x) = \frac{1}{N}\sum_{i=1}^{N} w^{(i)}\,b_\sigma\big(x - x^{(i)}\big).
$$

  → Widen each spike into a small bump and add them up to see the smooth density. Bump **width** matters more than shape: too wide smears detail, too narrow looks like spikes.

- **→ Intuition:** particles = a self-adapting, dimension-proof representation of *any* distribution. This is the data structure a particle filter carries and updates.

### 🧩 Summary

- The exact filter is the **Bayesian recursion**: predict via the Chapman–Kolmogorov integral, update via Bayes' rule — but the integrals are generally intractable.

- **Grid** integration suffers the **curse of dimensionality**; **Monte-Carlo** integration replaces it with sample averaging whose error is $\mathcal{O}(1/\sqrt N)$ **independent of dimension**.

- **Importance sampling** lets us sample from an easy $q$ and re-weight by $f/q$; **weight normalization** cancels the intractable evidence with a cheap 1-D sum.

- A pdf is stored as a **weighted sum of impulses** at particle locations (visualized with **kernels**), giving a dimension-proof representation of *any* density.

- These are exactly the pieces the particle filter assembles next.

---
*Next: [16 · The Particle Filter — SIS, Degeneracy & Resampling](16_Particle_Filter_SIS_Resampling.ipynb).*